# Mega Join: dataset bruto

Este notebook monta a base para prever o vencedor de cada corrida. O banco `formula.db` é aberto somente para leitura.

- Período: 2003 em diante.
- Uma linha por corrida e piloto, usando resultados do tipo `RACE_RESULT`.
- Fins de semana com sprint ficam fora, inclusive a corrida principal.
- As sessões de pista selecionadas viram colunas. Pit stops ficam fora.

Os joins preservam uma linha por corrida e piloto. O dataset sai em `dataset_completo.csv` e `dataset_completo.pkl`. Limpeza, leakage e features ficam para as proximas etapas.

## Por que o recorte comeca em 2003?

Treinos e classificacao ficam mais consistentes nesse periodo. Antes de 2003, pre-classificacao, grids maiores, carros compartilhados e participacao irregular tornam os resultados menos comparaveis aos atuais. A partir de 2003, o tamanho dos grids, o numero de equipes e pilotos e a estrutura de pontuacao ficam mais proximos das temporadas recentes. O recorte ainda mantem milhares de resultados para treinar e avaliar o modelo.

In [1]:
import sqlite3
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

# Le o banco sem alterar os dados.
DB_ORIGINAL = Path("formula.db")
conn = sqlite3.connect(f"file:{DB_ORIGINAL}?mode=ro", uri=True)
print(f"'{DB_ORIGINAL}' aberto em modo somente leitura.")

'formula.db' aberto em modo somente leitura.


## Fins de semana com sprint

O dataset exclui o fim de semana inteiro, incluindo o GP principal, para manter apenas o formato regular. O banco deixa `sprint_race_date` vazio em varios eventos de 2021 a 2023; por isso, a consulta verifica tambem formato, datas, voltas e distancias de sprint.
### Campos derivados de sprint

Alem de excluir os fins de semana sprint, nao exportamos estatisticas de sprint associadas a pilotos, construtores, fabricantes de motor ou pneus. Isso inclui melhor resultado, largadas e vitorias. Sao agregados de carreira da base e nao estao limitados ao que era conhecido antes de cada corrida; em linhas antigas podem incluir sprints futuros. As colunas descritivas do calendario sprint tambem nao precisam sair no dataset, pois servem apenas para identificar os eventos a excluir.


## Sessoes incluidas e excluidas

Resultados de treino, grid e classificacao selecionados viram colunas, mantendo uma linha por corrida e piloto. Pit stops podem ter varias linhas por piloto e corrida; junta-los diretamente duplicaria resultados e eles so acontecem durante a prova. Driver of the Day e decidido depois da corrida. Nenhum dos dois entra no resumo de sessoes ou nas features pre-corrida.

In [2]:
MEGA_JOIN_SQL = """
WITH sess AS (
    -- Resume sessoes em colunas. Pit stops e driver of the day ficam de fora.
    -- Remove fins de semana sprint usando todos os campos de sprint;
    -- sprint_race_date falta em varios eventos de 2021 a 2023.
    SELECT race_id, driver_id,
      MAX(CASE WHEN type='STARTING_GRID_POSITION'   THEN position_number END)        AS sess_grid_position_number,
      MAX(CASE WHEN type='STARTING_GRID_POSITION'   THEN position_text END)          AS sess_grid_position_text,
      MAX(CASE WHEN type='STARTING_GRID_POSITION'   THEN starting_grid_position_time_millis END) AS sess_grid_time_millis,
      MAX(CASE WHEN type='PRE_QUALIFYING_RESULT'    THEN position_number END)        AS sess_prequali_position_number,
      MAX(CASE WHEN type='PRE_QUALIFYING_RESULT'    THEN position_text END)          AS sess_prequali_position_text,
      MAX(CASE WHEN type='PRE_QUALIFYING_RESULT'    THEN qualifying_time_millis END) AS sess_prequali_time_millis,
      MAX(CASE WHEN type='FREE_PRACTICE_1_RESULT'   THEN position_number END)        AS sess_fp1_position_number,
      MAX(CASE WHEN type='FREE_PRACTICE_1_RESULT'   THEN position_text END)          AS sess_fp1_position_text,
      MAX(CASE WHEN type='FREE_PRACTICE_1_RESULT'   THEN practice_time_millis END)   AS sess_fp1_time_millis,
      MAX(CASE WHEN type='FREE_PRACTICE_1_RESULT'   THEN practice_laps END)          AS sess_fp1_laps,
      MAX(CASE WHEN type='FREE_PRACTICE_2_RESULT'   THEN position_number END)        AS sess_fp2_position_number,
      MAX(CASE WHEN type='FREE_PRACTICE_2_RESULT'   THEN position_text END)          AS sess_fp2_position_text,
      MAX(CASE WHEN type='FREE_PRACTICE_2_RESULT'   THEN practice_time_millis END)   AS sess_fp2_time_millis,
      MAX(CASE WHEN type='FREE_PRACTICE_2_RESULT'   THEN practice_laps END)          AS sess_fp2_laps,
      MAX(CASE WHEN type='FREE_PRACTICE_3_RESULT'   THEN position_number END)        AS sess_fp3_position_number,
      MAX(CASE WHEN type='FREE_PRACTICE_3_RESULT'   THEN position_text END)          AS sess_fp3_position_text,
      MAX(CASE WHEN type='FREE_PRACTICE_3_RESULT'   THEN practice_time_millis END)   AS sess_fp3_time_millis,
      MAX(CASE WHEN type='FREE_PRACTICE_3_RESULT'   THEN practice_laps END)          AS sess_fp3_laps,
      MAX(CASE WHEN type='FREE_PRACTICE_4_RESULT'   THEN position_number END)        AS sess_fp4_position_number,
      MAX(CASE WHEN type='FREE_PRACTICE_4_RESULT'   THEN position_text END)          AS sess_fp4_position_text,
      MAX(CASE WHEN type='FREE_PRACTICE_4_RESULT'   THEN practice_time_millis END)   AS sess_fp4_time_millis,
      MAX(CASE WHEN type='FREE_PRACTICE_4_RESULT'   THEN practice_laps END)          AS sess_fp4_laps,
      MAX(CASE WHEN type='WARMING_UP_RESULT'        THEN position_number END)        AS sess_warmup_position_number,
      MAX(CASE WHEN type='WARMING_UP_RESULT'        THEN position_text END)          AS sess_warmup_position_text,
      MAX(CASE WHEN type='WARMING_UP_RESULT'        THEN practice_time_millis END)   AS sess_warmup_time_millis,
      MAX(CASE WHEN type='WARMING_UP_RESULT'        THEN practice_laps END)          AS sess_warmup_laps,
      MAX(CASE WHEN type='QUALIFYING_RESULT'        THEN position_number END)        AS sess_quali_position_number,
      MAX(CASE WHEN type='QUALIFYING_RESULT'        THEN position_text END)          AS sess_quali_position_text,
      MAX(CASE WHEN type='QUALIFYING_RESULT'        THEN qualifying_time_millis END) AS sess_quali_time_millis,
      MAX(CASE WHEN type='QUALIFYING_RESULT'        THEN qualifying_q1_millis END)   AS sess_quali_q1_millis,
      MAX(CASE WHEN type='QUALIFYING_RESULT'        THEN qualifying_q2_millis END)   AS sess_quali_q2_millis,
      MAX(CASE WHEN type='QUALIFYING_RESULT'        THEN qualifying_q3_millis END)   AS sess_quali_q3_millis,
      MAX(CASE WHEN type='QUALIFYING_RESULT'        THEN qualifying_laps END)        AS sess_quali_laps,
      MAX(CASE WHEN type='QUALIFYING_1_RESULT'      THEN position_number END)        AS sess_quali1_position_number,
      MAX(CASE WHEN type='QUALIFYING_1_RESULT'      THEN position_text END)          AS sess_quali1_position_text,
      MAX(CASE WHEN type='QUALIFYING_1_RESULT'      THEN qualifying_time_millis END) AS sess_quali1_time_millis,
      MAX(CASE WHEN type='QUALIFYING_1_RESULT'      THEN qualifying_laps END)        AS sess_quali1_laps,
      MAX(CASE WHEN type='QUALIFYING_2_RESULT'      THEN position_number END)        AS sess_quali2_position_number,
      MAX(CASE WHEN type='QUALIFYING_2_RESULT'      THEN position_text END)          AS sess_quali2_position_text,
      MAX(CASE WHEN type='QUALIFYING_2_RESULT'      THEN qualifying_time_millis END) AS sess_quali2_time_millis,
      MAX(CASE WHEN type='QUALIFYING_2_RESULT'      THEN qualifying_laps END)        AS sess_quali2_laps,
      MAX(CASE WHEN type='FASTEST_LAP'              THEN fastest_lap_lap END)        AS sess_fastest_lap_lap,
      MAX(CASE WHEN type='FASTEST_LAP'              THEN fastest_lap_time_millis END) AS sess_fastest_lap_time_millis
    FROM race_data
    WHERE type IN ('STARTING_GRID_POSITION','PRE_QUALIFYING_RESULT',
                   'FREE_PRACTICE_1_RESULT','FREE_PRACTICE_2_RESULT',
                   'FREE_PRACTICE_3_RESULT','FREE_PRACTICE_4_RESULT',
                   'WARMING_UP_RESULT','QUALIFYING_RESULT','QUALIFYING_1_RESULT',
                   'QUALIFYING_2_RESULT','FASTEST_LAP')
    GROUP BY race_id, driver_id
),
ed AS (
    -- Junta equipes, chassis e motores usados por piloto e rodada.
    -- Assim, essas informacoes nao criam linhas extras.
    SELECT sed.year                       AS year,
           sed.driver_id                  AS driver_id,
           ra.round                       AS round,
           GROUP_CONCAT(DISTINCT sed.entrant_id)              AS ed_entrant_ids,
           GROUP_CONCAT(DISTINCT ent.name)                    AS ed_entrant_names,
           GROUP_CONCAT(DISTINCT sed.rounds_text)             AS ed_rounds_text,
           GROUP_CONCAT(DISTINCT sed.test_driver)             AS ed_test_driver,
           GROUP_CONCAT(DISTINCT sec.chassis_id)              AS chassis_ids,
           GROUP_CONCAT(DISTINCT ch.name)                     AS chassis_names,
           GROUP_CONCAT(DISTINCT ch.full_name)                AS chassis_full_names,
           GROUP_CONCAT(DISTINCT ch.constructor_id)           AS chassis_constructor_ids,
           GROUP_CONCAT(DISTINCT see.engine_id)               AS engine_ids,
           GROUP_CONCAT(DISTINCT eng.name)                    AS engine_names,
           GROUP_CONCAT(DISTINCT eng.full_name)               AS engine_full_names,
           GROUP_CONCAT(DISTINCT eng.capacity)                AS engine_capacities,
           GROUP_CONCAT(DISTINCT eng.configuration)           AS engine_configurations,
           GROUP_CONCAT(DISTINCT eng.aspiration)              AS engine_aspirations
    FROM season_entrant_driver sed
    JOIN race ra ON ra.year = sed.year
    LEFT JOIN season_entrant_chassis sec ON sec.year = sed.year AND sec.entrant_id = sed.entrant_id
    LEFT JOIN chassis ch   ON ch.id  = sec.chassis_id
    LEFT JOIN season_entrant_engine see ON see.year = sed.year AND see.entrant_id = sed.entrant_id
    LEFT JOIN engine eng   ON eng.id = see.engine_id
    LEFT JOIN entrant ent  ON ent.id = sed.entrant_id
    WHERE INSTR(';' || sed.rounds || ';', ';' || ra.round || ';') > 0
    GROUP BY sed.year, sed.driver_id, ra.round
),
cstand AS (
    -- Ha chaves repetidas nesta tabela. Usa uma linha por corrida e equipe
    -- para evitar resultados duplicados.
    SELECT race_id, constructor_id, position_display_order, position_number,
           position_text, engine_manufacturer_id, points, positions_gained,
           championship_won
    FROM (
        SELECT *, ROW_NUMBER() OVER (
                   PARTITION BY race_id, constructor_id
                   ORDER BY position_number) AS rn
        FROM race_constructor_standing
    )
    WHERE rn = 1
)
SELECT
       rd."race_id" AS rd_race_id,
       rd."type" AS rd_type,
       rd."position_display_order" AS rd_position_display_order,
       rd."position_number" AS rd_position_number,
       rd."position_text" AS rd_position_text,
       rd."driver_number" AS rd_driver_number,
       rd."driver_id" AS rd_driver_id,
       rd."constructor_id" AS rd_constructor_id,
       rd."engine_manufacturer_id" AS rd_engine_manufacturer_id,
       rd."tyre_manufacturer_id" AS rd_tyre_manufacturer_id,
       rd."practice_time" AS rd_practice_time,
       rd."practice_time_millis" AS rd_practice_time_millis,
       rd."practice_gap" AS rd_practice_gap,
       rd."practice_gap_millis" AS rd_practice_gap_millis,
       rd."practice_interval" AS rd_practice_interval,
       rd."practice_interval_millis" AS rd_practice_interval_millis,
       rd."practice_laps" AS rd_practice_laps,
       rd."qualifying_time" AS rd_qualifying_time,
       rd."qualifying_time_millis" AS rd_qualifying_time_millis,
       rd."qualifying_q1" AS rd_qualifying_q1,
       rd."qualifying_q1_millis" AS rd_qualifying_q1_millis,
       rd."qualifying_q2" AS rd_qualifying_q2,
       rd."qualifying_q2_millis" AS rd_qualifying_q2_millis,
       rd."qualifying_q3" AS rd_qualifying_q3,
       rd."qualifying_q3_millis" AS rd_qualifying_q3_millis,
       rd."qualifying_gap" AS rd_qualifying_gap,
       rd."qualifying_gap_millis" AS rd_qualifying_gap_millis,
       rd."qualifying_interval" AS rd_qualifying_interval,
       rd."qualifying_interval_millis" AS rd_qualifying_interval_millis,
       rd."qualifying_laps" AS rd_qualifying_laps,
       rd."starting_grid_position_qualification_position_number" AS rd_starting_grid_position_qualification_position_number,
       rd."starting_grid_position_qualification_position_text" AS rd_starting_grid_position_qualification_position_text,
       rd."starting_grid_position_grid_penalty" AS rd_starting_grid_position_grid_penalty,
       rd."starting_grid_position_grid_penalty_positions" AS rd_starting_grid_position_grid_penalty_positions,
       rd."starting_grid_position_time" AS rd_starting_grid_position_time,
       rd."starting_grid_position_time_millis" AS rd_starting_grid_position_time_millis,
       rd."race_shared_car" AS rd_race_shared_car,
       rd."race_laps" AS rd_race_laps,
       rd."race_time" AS rd_race_time,
       rd."race_time_millis" AS rd_race_time_millis,
       rd."race_time_penalty" AS rd_race_time_penalty,
       rd."race_time_penalty_millis" AS rd_race_time_penalty_millis,
       rd."race_gap" AS rd_race_gap,
       rd."race_gap_millis" AS rd_race_gap_millis,
       rd."race_gap_laps" AS rd_race_gap_laps,
       rd."race_interval" AS rd_race_interval,
       rd."race_interval_millis" AS rd_race_interval_millis,
       rd."race_reason_retired" AS rd_race_reason_retired,
       rd."race_points" AS rd_race_points,
       rd."race_pole_position" AS rd_race_pole_position,
       rd."race_qualification_position_number" AS rd_race_qualification_position_number,
       rd."race_qualification_position_text" AS rd_race_qualification_position_text,
       rd."race_grid_position_number" AS rd_race_grid_position_number,
       rd."race_grid_position_text" AS rd_race_grid_position_text,
       rd."race_positions_gained" AS rd_race_positions_gained,
       rd."race_pit_stops" AS rd_race_pit_stops,
       rd."race_fastest_lap" AS rd_race_fastest_lap,
       rd."race_driver_of_the_day" AS rd_race_driver_of_the_day,
       rd."race_grand_slam" AS rd_race_grand_slam,
       rd."fastest_lap_lap" AS rd_fastest_lap_lap,
       rd."fastest_lap_time" AS rd_fastest_lap_time,
       rd."fastest_lap_time_millis" AS rd_fastest_lap_time_millis,
       rd."fastest_lap_gap" AS rd_fastest_lap_gap,
       rd."fastest_lap_gap_millis" AS rd_fastest_lap_gap_millis,
       rd."fastest_lap_interval" AS rd_fastest_lap_interval,
       rd."fastest_lap_interval_millis" AS rd_fastest_lap_interval_millis,
       rd."pit_stop_stop" AS rd_pit_stop_stop,
       rd."pit_stop_lap" AS rd_pit_stop_lap,
       rd."pit_stop_time" AS rd_pit_stop_time,
       rd."pit_stop_time_millis" AS rd_pit_stop_time_millis,
       rd."driver_of_the_day_percentage" AS rd_driver_of_the_day_percentage,
       r."id" AS race_id,
       r."year" AS race_year,
       r."round" AS race_round,
       r."date" AS race_date,
       r."time" AS race_time,
       r."grand_prix_id" AS race_grand_prix_id,
       r."official_name" AS race_official_name,
       r."qualifying_format" AS race_qualifying_format,
       r."circuit_id" AS race_circuit_id,
       r."circuit_layout_id" AS race_circuit_layout_id,
       r."circuit_type" AS race_circuit_type,
       r."direction" AS race_direction,
       r."course_length" AS race_course_length,
       r."turns" AS race_turns,
       r."laps" AS race_laps,
       r."distance" AS race_distance,
       r."scheduled_laps" AS race_scheduled_laps,
       r."scheduled_distance" AS race_scheduled_distance,
       r."drivers_championship_decider" AS race_drivers_championship_decider,
       r."constructors_championship_decider" AS race_constructors_championship_decider,
       r."pre_qualifying_date" AS race_pre_qualifying_date,
       r."pre_qualifying_time" AS race_pre_qualifying_time,
       r."free_practice_1_date" AS race_free_practice_1_date,
       r."free_practice_1_time" AS race_free_practice_1_time,
       r."free_practice_2_date" AS race_free_practice_2_date,
       r."free_practice_2_time" AS race_free_practice_2_time,
       r."free_practice_3_date" AS race_free_practice_3_date,
       r."free_practice_3_time" AS race_free_practice_3_time,
       r."free_practice_4_date" AS race_free_practice_4_date,
       r."free_practice_4_time" AS race_free_practice_4_time,
       r."qualifying_1_date" AS race_qualifying_1_date,
       r."qualifying_1_time" AS race_qualifying_1_time,
       r."qualifying_2_date" AS race_qualifying_2_date,
       r."qualifying_2_time" AS race_qualifying_2_time,
       r."qualifying_date" AS race_qualifying_date,
       r."qualifying_time" AS race_qualifying_time,
       r."warming_up_date" AS race_warming_up_date,
       r."warming_up_time" AS race_warming_up_time,
       cir."id" AS circuit_id,
       cir."name" AS circuit_name,
       cir."full_name" AS circuit_full_name,
       cir."previous_names" AS circuit_previous_names,
       cir."type" AS circuit_type,
       cir."direction" AS circuit_direction,
       cir."place_name" AS circuit_place_name,
       cir."country_id" AS circuit_country_id,
       cir."latitude" AS circuit_latitude,
       cir."longitude" AS circuit_longitude,
       cir."length" AS circuit_length,
       cir."turns" AS circuit_turns,
       cir."total_races_held" AS circuit_total_races_held,
       lay."id" AS layout_id,
       lay."circuit_id" AS layout_circuit_id,
       lay."effective" AS layout_effective,
       lay."length" AS layout_length,
       lay."turns" AS layout_turns,
       gp."id" AS gp_id,
       gp."name" AS gp_name,
       gp."full_name" AS gp_full_name,
       gp."short_name" AS gp_short_name,
       gp."abbreviation" AS gp_abbreviation,
       gp."country_id" AS gp_country_id,
       gp."total_races_held" AS gp_total_races_held,
       hco."id" AS gp_country_id,
       hco."alpha2_code" AS gp_country_alpha2_code,
       hco."alpha3_code" AS gp_country_alpha3_code,
       hco."ioc_code" AS gp_country_ioc_code,
       hco."name" AS gp_country_name,
       hco."demonym" AS gp_country_demonym,
       hco."continent_id" AS gp_country_continent_id,
       hcn."id" AS gp_continent_id,
       hcn."code" AS gp_continent_code,
       hcn."name" AS gp_continent_name,
       hcn."demonym" AS gp_continent_demonym,
       d."id" AS driver_id,
       d."name" AS driver_name,
       d."first_name" AS driver_first_name,
       d."last_name" AS driver_last_name,
       d."full_name" AS driver_full_name,
       d."abbreviation" AS driver_abbreviation,
       d."permanent_number" AS driver_permanent_number,
       d."gender" AS driver_gender,
       d."date_of_birth" AS driver_date_of_birth,
       d."date_of_death" AS driver_date_of_death,
       d."place_of_birth" AS driver_place_of_birth,
       d."country_of_birth_country_id" AS driver_country_of_birth_country_id,
       d."nationality_country_id" AS driver_nationality_country_id,
       d."second_nationality_country_id" AS driver_second_nationality_country_id,
       d."best_championship_position" AS driver_best_championship_position,
       d."best_starting_grid_position" AS driver_best_starting_grid_position,
       d."best_race_result" AS driver_best_race_result,
       d."total_championship_wins" AS driver_total_championship_wins,
       d."total_race_entries" AS driver_total_race_entries,
       d."total_race_starts" AS driver_total_race_starts,
       d."total_race_wins" AS driver_total_race_wins,
       d."total_race_laps" AS driver_total_race_laps,
       d."total_podiums" AS driver_total_podiums,
       d."total_points" AS driver_total_points,
       d."total_championship_points" AS driver_total_championship_points,
       d."total_pole_positions" AS driver_total_pole_positions,
       d."total_fastest_laps" AS driver_total_fastest_laps,
       d."total_driver_of_the_day" AS driver_total_driver_of_the_day,
       d."total_grand_slams" AS driver_total_grand_slams,
       nco."id" AS nat_country_id,
       nco."alpha2_code" AS nat_country_alpha2_code,
       nco."alpha3_code" AS nat_country_alpha3_code,
       nco."ioc_code" AS nat_country_ioc_code,
       nco."name" AS nat_country_name,
       nco."demonym" AS nat_country_demonym,
       nco."continent_id" AS nat_country_continent_id,
       ncn."id" AS nat_continent_id,
       ncn."code" AS nat_continent_code,
       ncn."name" AS nat_continent_name,
       ncn."demonym" AS nat_continent_demonym,
       bco."id" AS birth_country_id,
       bco."alpha2_code" AS birth_country_alpha2_code,
       bco."alpha3_code" AS birth_country_alpha3_code,
       bco."ioc_code" AS birth_country_ioc_code,
       bco."name" AS birth_country_name,
       bco."demonym" AS birth_country_demonym,
       bco."continent_id" AS birth_country_continent_id,
       bcn."id" AS birth_continent_id,
       bcn."code" AS birth_continent_code,
       bcn."name" AS birth_continent_name,
       bcn."demonym" AS birth_continent_demonym,
       ct."id" AS constructor_id,
       ct."name" AS constructor_name,
       ct."full_name" AS constructor_full_name,
       ct."country_id" AS constructor_country_id,
       ct."best_championship_position" AS constructor_best_championship_position,
       ct."best_starting_grid_position" AS constructor_best_starting_grid_position,
       ct."best_race_result" AS constructor_best_race_result,
       ct."total_championship_wins" AS constructor_total_championship_wins,
       ct."total_race_entries" AS constructor_total_race_entries,
       ct."total_race_starts" AS constructor_total_race_starts,
       ct."total_race_wins" AS constructor_total_race_wins,
       ct."total_1_and_2_finishes" AS constructor_total_1_and_2_finishes,
       ct."total_race_laps" AS constructor_total_race_laps,
       ct."total_podiums" AS constructor_total_podiums,
       ct."total_podium_races" AS constructor_total_podium_races,
       ct."total_points" AS constructor_total_points,
       ct."total_championship_points" AS constructor_total_championship_points,
       ct."total_pole_positions" AS constructor_total_pole_positions,
       ct."total_fastest_laps" AS constructor_total_fastest_laps,
       cco."id" AS constructor_country_id,
       cco."alpha2_code" AS constructor_country_alpha2_code,
       cco."alpha3_code" AS constructor_country_alpha3_code,
       cco."ioc_code" AS constructor_country_ioc_code,
       cco."name" AS constructor_country_name,
       cco."demonym" AS constructor_country_demonym,
       cco."continent_id" AS constructor_country_continent_id,
       ccn."id" AS constructor_continent_id,
       ccn."code" AS constructor_continent_code,
       ccn."name" AS constructor_continent_name,
       ccn."demonym" AS constructor_continent_demonym,
       em."id" AS engmfr_id,
       em."name" AS engmfr_name,
       em."country_id" AS engmfr_country_id,
       em."best_championship_position" AS engmfr_best_championship_position,
       em."best_starting_grid_position" AS engmfr_best_starting_grid_position,
       em."best_race_result" AS engmfr_best_race_result,
       em."total_championship_wins" AS engmfr_total_championship_wins,
       em."total_race_entries" AS engmfr_total_race_entries,
       em."total_race_starts" AS engmfr_total_race_starts,
       em."total_race_wins" AS engmfr_total_race_wins,
       em."total_race_laps" AS engmfr_total_race_laps,
       em."total_podiums" AS engmfr_total_podiums,
       em."total_podium_races" AS engmfr_total_podium_races,
       em."total_points" AS engmfr_total_points,
       em."total_championship_points" AS engmfr_total_championship_points,
       em."total_pole_positions" AS engmfr_total_pole_positions,
       em."total_fastest_laps" AS engmfr_total_fastest_laps,
       tm."id" AS tyremfr_id,
       tm."name" AS tyremfr_name,
       tm."country_id" AS tyremfr_country_id,
       tm."best_starting_grid_position" AS tyremfr_best_starting_grid_position,
       tm."best_race_result" AS tyremfr_best_race_result,
       tm."total_race_entries" AS tyremfr_total_race_entries,
       tm."total_race_starts" AS tyremfr_total_race_starts,
       tm."total_race_wins" AS tyremfr_total_race_wins,
       tm."total_race_laps" AS tyremfr_total_race_laps,
       tm."total_podiums" AS tyremfr_total_podiums,
       tm."total_podium_races" AS tyremfr_total_podium_races,
       tm."total_pole_positions" AS tyremfr_total_pole_positions,
       tm."total_fastest_laps" AS tyremfr_total_fastest_laps,
       rds."position_display_order" AS dstand_position_display_order,
       rds."position_number" AS dstand_position_number,
       rds."position_text" AS dstand_position_text,
       rds."points" AS dstand_points,
       rds."positions_gained" AS dstand_positions_gained,
       rds."championship_won" AS dstand_championship_won,
       cstand."position_display_order" AS cstand_position_display_order,
       cstand."position_number" AS cstand_position_number,
       cstand."position_text" AS cstand_position_text,
       cstand."engine_manufacturer_id" AS cstand_engine_manufacturer_id,
       cstand."points" AS cstand_points,
       cstand."positions_gained" AS cstand_positions_gained,
       cstand."championship_won" AS cstand_championship_won,
       ed."ed_entrant_ids",
       ed."ed_entrant_names",
       ed."ed_rounds_text",
       ed."ed_test_driver",
       ed."chassis_ids",
       ed."chassis_names",
       ed."chassis_full_names",
       ed."chassis_constructor_ids",
       ed."engine_ids",
       ed."engine_names",
       ed."engine_full_names",
       ed."engine_capacities",
       ed."engine_configurations",
       ed."engine_aspirations",
       sess."sess_grid_position_number",
       sess."sess_grid_position_text",
       sess."sess_grid_time_millis",
       sess."sess_prequali_position_number",
       sess."sess_prequali_position_text",
       sess."sess_prequali_time_millis",
       sess."sess_fp1_position_number",
       sess."sess_fp1_position_text",
       sess."sess_fp1_time_millis",
       sess."sess_fp1_laps",
       sess."sess_fp2_position_number",
       sess."sess_fp2_position_text",
       sess."sess_fp2_time_millis",
       sess."sess_fp2_laps",
       sess."sess_fp3_position_number",
       sess."sess_fp3_position_text",
       sess."sess_fp3_time_millis",
       sess."sess_fp3_laps",
       sess."sess_fp4_position_number",
       sess."sess_fp4_position_text",
       sess."sess_fp4_time_millis",
       sess."sess_fp4_laps",
       sess."sess_warmup_position_number",
       sess."sess_warmup_position_text",
       sess."sess_warmup_time_millis",
       sess."sess_warmup_laps",
       sess."sess_quali_position_number",
       sess."sess_quali_position_text",
       sess."sess_quali_time_millis",
       sess."sess_quali_q1_millis",
       sess."sess_quali_q2_millis",
       sess."sess_quali_q3_millis",
       sess."sess_quali_laps",
       sess."sess_quali1_position_number",
       sess."sess_quali1_position_text",
       sess."sess_quali1_time_millis",
       sess."sess_quali1_laps",
       sess."sess_quali2_position_number",
       sess."sess_quali2_position_text",
       sess."sess_quali2_time_millis",
       sess."sess_quali2_laps",
       sess."sess_fastest_lap_lap",
       sess."sess_fastest_lap_time_millis"
FROM race_data rd
JOIN race r ON r.id = rd.race_id
LEFT JOIN sess ON sess.race_id = rd.race_id AND sess.driver_id = rd.driver_id
LEFT JOIN circuit cir        ON cir.id = r.circuit_id
LEFT JOIN circuit_layout lay ON lay.id = r.circuit_layout_id
LEFT JOIN grand_prix gp      ON gp.id  = r.grand_prix_id
LEFT JOIN country hco        ON hco.id = gp.country_id
LEFT JOIN continent hcn      ON hcn.id = hco.continent_id
LEFT JOIN driver d           ON d.id   = rd.driver_id
LEFT JOIN country nco        ON nco.id = d.nationality_country_id
LEFT JOIN continent ncn      ON ncn.id = nco.continent_id
LEFT JOIN country bco        ON bco.id = d.country_of_birth_country_id
LEFT JOIN continent bcn      ON bcn.id = bco.continent_id
LEFT JOIN constructor ct     ON ct.id  = rd.constructor_id
LEFT JOIN country cco        ON cco.id = ct.country_id
LEFT JOIN continent ccn      ON ccn.id = cco.continent_id
LEFT JOIN engine_manufacturer em ON em.id = rd.engine_manufacturer_id
LEFT JOIN tyre_manufacturer tm   ON tm.id = rd.tyre_manufacturer_id
LEFT JOIN ed   ON ed.year = r.year AND ed.driver_id = rd.driver_id AND ed.round = r.round
LEFT JOIN race_driver_standing rds ON rds.race_id = rd.race_id AND rds.driver_id = rd.driver_id
LEFT JOIN cstand ON cstand.race_id = rd.race_id AND cstand.constructor_id = rd.constructor_id
WHERE rd.type = 'RACE_RESULT'
   AND r.year >= 2003
   AND r.sprint_qualifying_format IS NULL
   AND r.sprint_qualifying_date IS NULL
   AND r.sprint_race_date IS NULL
   AND r.sprint_race_laps IS NULL
   AND r.sprint_race_distance IS NULL
   AND r.sprint_race_scheduled_laps IS NULL
   AND r.sprint_race_scheduled_distance IS NULL
"""

In [3]:
df = pd.read_sql_query(MEGA_JOIN_SQL, conn)

print(f"shape: {df.shape[0]:,} linhas × {df.shape[1]} colunas")

shape: 9,146 linhas × 326 colunas


In [4]:
# Confere o resultado do join; nao altera os dados.
sprint_predicate = """
sprint_qualifying_format IS NOT NULL
OR sprint_qualifying_date IS NOT NULL
OR sprint_race_date IS NOT NULL
OR sprint_race_laps IS NOT NULL
OR sprint_race_distance IS NOT NULL
OR sprint_race_scheduled_laps IS NOT NULL
OR sprint_race_scheduled_distance IS NOT NULL
"""
sprints_excluidos = conn.execute(
    f"SELECT COUNT(*) FROM race WHERE {sprint_predicate}"
).fetchone()[0]
resultados_sprint_excluidos = conn.execute(
    "SELECT COUNT(*) FROM race_data rd JOIN race r ON r.id = rd.race_id "
    f"WHERE rd.type = 'RACE_RESULT' AND ({sprint_predicate})"
).fetchone()[0]
print(f"eventos sprint identificados: {sprints_excluidos} | linhas RACE_RESULT removidas: {resultados_sprint_excluidos:,}")

print("== Grão ==")
dups = df[df.duplicated(subset=['rd_race_id', 'rd_driver_id'], keep=False)]
n_corridas = df['rd_race_id'].nunique()
vencedores_multiplas = (df[df['rd_position_number'] == 1]
                        .groupby('rd_race_id').size() > 1).sum()
print(f"linhas: {len(df):,} | corridas: {n_corridas:,} | pilotos únicos: {df['rd_driver_id'].nunique()}")
print(f"linhas duplicadas (corrida, piloto): {len(dups)} "
      f"— carros compartilhados/duplas entradas da era antiga")
print(f"corridas com mais de um vencedor (position_number=1): {vencedores_multiplas}")

print("\n== Anos ==")
print(f"de {df['race_year'].min()} a {df['race_year'].max()} | somente temporadas 2003+")

print("\n== Cobertura de colunas-chave ==")
for coluna, rotulo in [
    ('chassis_ids', 'chassi via entrant'),
    ('engine_ids', 'motor via entrant'),
    ('ed_entrant_ids', 'entrant'),
    ('rd_race_grid_position_number', 'posição de grid (spine)'),
    ('sess_grid_position_number', 'grid via pivot'),
    ('sess_quali_time_millis', 'tempo de quali via pivot'),
    ('sess_fp1_time_millis', 'tempo de FP1 via pivot'),
    ('sess_fastest_lap_time_millis', 'volta mais rápida via pivot'),
    ('dstand_points', 'standing de pilotos'),
    ('cstand_points', 'standing de constructors'),
]:
    n = df[coluna].notna().sum()
    print(f"  {rotulo:<32} {n:>7,}/{len(df):,} ({n / len(df):.1%})")

print("\n== Target ==")
print((df['rd_position_number'] == 1).value_counts().rename({True: 'venceu', False: 'não venceu'}))

dups.sort_values(['rd_race_id', 'rd_driver_id']).head(8)

eventos sprint identificados: 30 | linhas RACE_RESULT removidas: 590
== Grão ==
linhas: 9,146 | corridas: 436 | pilotos únicos: 117
linhas duplicadas (corrida, piloto): 0 — carros compartilhados/duplas entradas da era antiga
corridas com mais de um vencedor (position_number=1): 0

== Anos ==
de 2003 a 2026 | somente temporadas 2003+

== Cobertura de colunas-chave ==
  chassi via entrant                 9,146/9,146 (100.0%)
  motor via entrant                  9,146/9,146 (100.0%)
  entrant                            9,146/9,146 (100.0%)
  posição de grid (spine)            8,939/9,146 (97.7%)
  grid via pivot                     9,008/9,146 (98.5%)
  tempo de quali via pivot           1,002/9,146 (11.0%)
  tempo de FP1 via pivot             8,251/9,146 (90.2%)
  volta mais rápida via pivot        8,746/9,146 (95.6%)
  standing de pilotos                8,945/9,146 (97.8%)
  standing de constructors           9,092/9,146 (99.4%)

== Target ==
rd_position_number
não venceu    8710
venceu

,rd_race_id,rd_type,rd_position_display_order,rd_position_number,rd_position_text,rd_driver_number,rd_driver_id,rd_constructor_id,rd_engine_manufacturer_id,rd_tyre_manufacturer_id,rd_practice_time,rd_practice_time_millis,rd_practice_gap,rd_practice_gap_millis,rd_practice_interval,rd_practice_interval_millis,rd_practice_laps,rd_qualifying_time,rd_qualifying_time_millis,rd_qualifying_q1,rd_qualifying_q1_millis,rd_qualifying_q2,rd_qualifying_q2_millis,rd_qualifying_q3,rd_qualifying_q3_millis,rd_qualifying_gap,rd_qualifying_gap_millis,rd_qualifying_interval,rd_qualifying_interval_millis,rd_qualifying_laps,rd_starting_grid_position_qualification_position_number,rd_starting_grid_position_qualification_position_text,rd_starting_grid_position_grid_penalty,rd_starting_grid_position_grid_penalty_positions,rd_starting_grid_position_time,rd_starting_grid_position_time_millis,rd_race_shared_car,rd_race_laps,rd_race_time,rd_race_time_millis,rd_race_time_penalty,rd_race_time_penalty_millis,rd_race_gap,rd_race_gap_millis,rd_race_gap_laps,rd_race_interval,rd_race_interval_millis,rd_race_reason_retired,rd_race_points,rd_race_pole_position,rd_race_qualification_position_number,rd_race_qualification_position_text,rd_race_grid_position_number,rd_race_grid_position_text,rd_race_positions_gained,rd_race_pit_stops,rd_race_fastest_lap,rd_race_driver_of_the_day,rd_race_grand_slam,rd_fastest_lap_lap,rd_fastest_lap_time,rd_fastest_lap_time_millis,rd_fastest_lap_gap,rd_fastest_lap_gap_millis,rd_fastest_lap_interval,rd_fastest_lap_interval_millis,rd_pit_stop_stop,rd_pit_stop_lap,rd_pit_stop_time,rd_pit_stop_time_millis,rd_driver_of_the_day_percentage,race_id,race_year,race_round,race_date,race_time,race_grand_prix_id,race_official_name,race_qualifying_format,race_circuit_id,race_circuit_layout_id,race_circuit_type,race_direction,race_course_length,race_turns,race_laps,race_distance,race_scheduled_laps,race_scheduled_distance,race_drivers_championship_decider,race_constructors_championship_decider,race_pre_qualifying_date,race_pre_qualifying_time,race_free_practice_1_date,race_free_practice_1_time,race_free_practice_2_date,race_free_practice_2_time,race_free_practice_3_date,race_free_practice_3_time,race_free_practice_4_date,race_free_practice_4_time,race_qualifying_1_date,race_qualifying_1_time,race_qualifying_2_date,race_qualifying_2_time,race_qualifying_date,race_qualifying_time,race_warming_up_date,race_warming_up_time,circuit_id,circuit_name,circuit_full_name,circuit_previous_names,circuit_type,circuit_direction,circuit_place_name,circuit_country_id,circuit_latitude,circuit_longitude,circuit_length,circuit_turns,circuit_total_races_held,layout_id,layout_circuit_id,layout_effective,layout_length,layout_turns,gp_id,gp_name,gp_full_name,gp_short_name,gp_abbreviation,gp_country_id,gp_total_races_held,gp_country_id,gp_country_alpha2_code,gp_country_alpha3_code,gp_country_ioc_code,gp_country_name,gp_country_demonym,gp_country_continent_id,gp_continent_id,gp_continent_code,gp_continent_name,gp_continent_demonym,driver_id,driver_name,driver_first_name,driver_last_name,driver_full_name,driver_abbreviation,driver_permanent_number,driver_gender,driver_date_of_birth,driver_date_of_death,driver_place_of_birth,driver_country_of_birth_country_id,driver_nationality_country_id,driver_second_nationality_country_id,driver_best_championship_position,driver_best_starting_grid_position,driver_best_race_result,driver_total_championship_wins,driver_total_race_entries,driver_total_race_starts,driver_total_race_wins,driver_total_race_laps,driver_total_podiums,driver_total_points,driver_total_championship_points,driver_total_pole_positions,driver_total_fastest_laps,driver_total_driver_of_the_day,driver_total_grand_slams,nat_country_id,nat_country_alpha2_code,nat_country_alpha3_code,nat_country_ioc_code,nat_country_name,nat_country_demonym,nat_country_continent_id,nat_continent_id,nat_continent_code,nat_continent_name,nat_continent_demonym,birth_country_id,birth_country_alpha2_co

In [5]:
csv_path = Path("dataset_completo.csv")
pkl_path = Path("dataset_completo.pkl")

df.to_csv(csv_path, index=False)
df.to_pickle(pkl_path)

for caminho in (csv_path, pkl_path):
    print(f"{caminho}: {caminho.stat().st_size / 1e6:.1f} MB")

dataset_completo.csv: 15.2 MB
dataset_completo.pkl: 24.0 MB


## Proximos passos

1. Limpar duplicatas e colunas sem dados.
2. Revisar leakage antes de escolher as features.
3. Criar features historicas e de pre-largada.
4. Separar treino ate 2024, validacao em 2025 e teste em 2026.